# Final Project

Author: Evan Whitfield

Course: ST 554

Purpose: Final Project

## Part 1 - Fitting Your Model

First, we need to import all the necessary modules for our task. There are several modules in `pyspark.ml` that we will need for various tasks.

In [3]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator


We need to read in our data for our machine learning process. This will serve as our training set for our model building process.

In [4]:
power_ml_data = pd.read_csv('power_ml_data.csv')

Time to start the spark session and turn the data into a spark dataframe.

In [5]:
spark_sess = SparkSession.builder.getOrCreate()

spark_df = spark_sess.createDataFrame(power_ml_data)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 22:54:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 22:54:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 22:54:45 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [6]:
spark_df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

We can see how the data looks and how it is stored for the different variable.

In [7]:
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In order to do some of the transformations we need, the `Hour` variable needs to be a Double type instead of a Long type. The schema shows us that it is currently a `long` type, which will not work appropriately. Therefore, we will cast is as a `Double` type below.

In [8]:
cast_hour_double = SQLTransformer(statement = 
    """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double
    FROM __THIS__
    """
)

Next, we want to change our hour variable into a binary for either day or night time using a threshold of 6.5.

In [9]:
binary_day_night = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="day_night")

We want to One Hot Encode the month variable. To

In [10]:
one_hot = OneHotEncoder(
    inputCols = ["Month"],
    outputCols = ["Month_ohe"],
    dropLast = False
)

Time to assemble the different weather related variables into a single column.

In [11]:
weather_assembled = VectorAssembler(
    inputCols = ["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"],
    outputCol = "weather",
    handleInvalid = "skip"
)

The next transform we want to do is a PCA with k = 2 and store the results in a new column.

In [12]:
pca = PCA(
    k = 2,
    inputCol = "weather",
    outputCol = "pca_features"
)

We need to set up our label and features column to get ready for the Linear Regression.

In [13]:
label_maker = SQLTransformer(statement =
    """
    SELECT *, Power_Zone_3 AS label
    FROM __THIS__
    """
)

In [14]:
features_assembler = VectorAssembler(
    inputCols = ["pca_features","day_night","Power_Zone_1","Power_Zone_2","Month_ohe"],
    outputCol = "features",
    handleInvalid = "skip"
)

In [15]:
lr = LinearRegression()

Next, we want to set up our grid of possible parameters to test to check to see what will produce the lowest RMSE.

In [16]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

Time to put all of our different transforms into a pipeline, ending with the Linear Regression.

In [17]:
pipeline = Pipeline(stages = [
    cast_hour_double, 
    binary_day_night, 
    one_hot, 
    weather_assembled,
    pca,
    label_maker,
    features_assembler,
    lr
])

The pipeline will be used as our estimator with 5 folds. We want to determine which set of parameters gives us the lowest RMSE.

In [18]:
cv = CrossValidator(
    estimator = pipeline,
    estimatorParamMaps = paramGrid,
    evaluator = RegressionEvaluator(metricName = 'rmse'),
    numFolds = 5,
    parallelism = 8
)

In [20]:
cvModel = cv.fit(spark_df)

26/04/30 23:00:03 WARN Instrumentation: [210c2ac0] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 23:00:03 WARN Instrumentation: [0a46b7c5] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 23:00:03 WARN Instrumentation: [20f562ce] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 23:00:03 WARN Instrumentation: [6f1c85f3] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 23:00:03 WARN Instrumentation: [5a8cc6f1] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 23:00:03 WARN Instrumentation: [210c2ac0] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 23:00:03 WARN Instrumentation: [12a319e0] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 23:00:03 WARN Instrumentation: [d92f6bee] regParam is zero, which might cause numerical ins

We need to do use a couple of attributes to reside to produce the best parameters for reducing the RMSE.

In [21]:
best_pipeline_model = cvModel.bestModel.stages[-1]

print("Best regParam:", best_pipeline_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_pipeline_model._java_obj.getElasticNetParam())

Best regParam: 0.5
Best elasticNetParam: 0.99


The best parameters for this run are shown above. I have gotten different results before, assuming due to the randomness of the folds, so I'm not sure if they are the absolute best the majority of the time. The RMSE below is similar to the results I have gotten before.

In [22]:
rmse_values = cvModel.avgMetrics
best_rmse = min(rmse_values)

print("Best CV RMSE:", best_rmse)

Best CV RMSE: 2147.8197685258865


The best CV RMSE was aroud 2148. Standardizing the values of the dataframe would make this value MUCH smaller. Unforunately, we did not do that here.

In [23]:
preds = cvModel.transform(spark_df)

In [24]:
preds_short = preds.select("label", "features", "prediction")

In [25]:
training_error = RegressionEvaluator().evaluate(cvModel.transform(spark_df))
print("RMSE with training error:", training_error)

RMSE with training error: 2147.0991651344534


If we look at the RMSE for the entire data set, it is very similar to the RMSE found in our CV.

In [26]:
preds_with_resids = preds_short.withColumn("residual",
    col("label") - col("prediction")
)

We want to see what the residuals look like. The residuals are very large, but small compared to our label and prediction columns. Again, standardizing would have kept the residuals much smaller. This probably should have been for modeling purposes because they non-standardized values have most likely have potential to mess with the PCA results or the Elastic net parameter since variables with larger values tend to be used more often in these models.

In [27]:
preds_with_resids.show(20)

+-----------+--------------------+------------------+------------------+
|      label|            features|        prediction|          residual|
+-----------+--------------------+------------------+------------------+
|20240.96386|(18,[0,1,3,4,6],[...|20877.477304233165|-636.5134442331655|
|20131.08434|(18,[0,1,3,4,6],[...| 18657.51706637134|1473.5672736286615|
|19668.43373|(18,[0,1,3,4,6],[...|18202.183165334594|1466.2505646654063|
|18899.27711|(18,[0,1,3,4,6],[...| 17588.32960498681|1310.9475050131878|
|18442.40964|(18,[0,1,3,4,6],[...|16995.170571225575|1447.2390687744264|
|18130.12048|(18,[0,1,3,4,6],[...|16515.731359439917|1614.3891205600848|
|17945.06024|(18,[0,1,3,4,6],[...|16091.454373523138|1853.6058664768607|
|17459.27711|(18,[0,1,3,4,6],[...| 15721.03136963183|1738.2457403681692|
|17025.54217|(18,[0,1,3,4,6],[...|15269.547544764268| 1755.994625235733|
|16794.21687|(18,[0,1,3,4,6],[...|14936.963131222084|1857.2537387779157|
|16638.07229|(18,[0,1,3,4,6],[...|14651.19597616853

## Part 2 - Streaming Part

We want to get out schema from our training dataframe and be prepared to use it in our streaming data.

In [28]:
schema = spark_df.schema

In [29]:
stream_df = spark_sess.readStream \
    .format("csv") \
    .option("header", True) \
    .schema(schema) \
    .load("Final Project Output")

The stream will look for the CSV files in the directoy "Final Project Output" that our .py file will create. It will use the same schema as our training data.

In [30]:
stream_preds = cvModel.transform(stream_df)

stream_residuals = stream_preds.withColumn(
    "residual", col("label") - col("prediction")
    ).select(
        "label",
        "prediction",
        "residual"
)

We want to create the same columns as we did for the training dataframe: `label`, `prediction`, and `residual`, but not the features.

In [33]:
label_stream = SQLTransformer(statement = 
    """
    SELECT 
        Power_Zone_3 AS label,
        Power_Zone_2,
        Power_Zone_1
    FROM __THIS__
    """
    ).transform(stream_df)

In [34]:
joined_stream = stream_residuals.join(
    label_stream,
    on = "label",
    how = "inner"
)

In [43]:
query = joined_stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

26/04/30 23:24:05 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-becf85be-4359-46cc-8ec0-dead18579c48. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 23:24:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+------------+------------+
|      label|        prediction|           residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+-------------------+------------+------------+
|9477.446809|  9774.18214363872| -296.7353346387208| 17350.20747| 25075.53611|
|37551.34796| 33327.80796341553|   4223.53999658447| 33300.95037| 48899.26748|
| 18201.1336|14998.963950945687| 3202.1696490543145| 18219.19505| 31022.16393|
|19055.54859| 21062.21040724402|-2006.6618172440212| 18775.50158| 28320.53274|
|13212.40122| 13763.10263757823| -550.7014175782297| 21394.60581| 34975.92998|
+-----------+------------------+-------------------+------------+------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+------------+------------+
|      label|        prediction|           residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+-------------------+------------+------------+
|9231.212485| 11018.32875481326|-1787.1162698132594| 25866.83032| 31318.63118|
|27138.80878|25379.867853274798| 1758.9409267252013| 24223.02006| 34483.28524|
|16641.33891| 21144.88859741239| -4503.549687412393| 17692.40506| 26191.09635|
|21309.67742| 18112.47680458306| 3197.2006154169394|  19159.7561| 33824.68085|
|21672.72727| 20637.79110290373|  1034.936167096268| 18740.52953| 35440.43057|
+-----------+------------------+-------------------+------------+------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+------------+------------+
|      label|        prediction|           residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+-------------------+------------+------------+
|18096.19433|15004.082453410072| 3092.1118765899264| 18538.69969| 31148.06557|
|20892.50255|21850.952631338765|  -958.450081338764| 26034.51143| 43690.61947|
|25339.35484|26171.432537407472| -832.0776974074724| 26484.14634| 44780.93617|
|21672.72727| 20637.79110290373|  1034.936167096268| 18740.52953| 35440.43057|
|21672.72727| 20637.79110290373|  1034.936167096268| 18740.52953| 35440.43057|
|21672.72727| 20637.79110290373|  1034.936167096268| 18740.52953| 35440.43057|
|16326.86415|16327.242376799699|-0.3782267996994051| 23516.00832| 37777.69912|
+-----------+------------------+-------------------+------------+------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+------------------+------------+------------+
|      label|        prediction|          residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+------------------+------------+------------+
|15260.79027|12968.299465920556|2292.4908040794435| 21110.78838| 32581.18162|
|11680.19208|11233.811687534822| 446.3803924651784| 22133.16968| 27011.40684|
|14324.21053|14108.589970514857|215.62055948514353| 14180.80495| 24022.03279|
|28733.72385| 27095.27642571877|1638.4474242812285| 24801.26582|  36071.7608|
|15323.22581|16544.395861490626|-1221.170051490626|     19650.0| 31692.25532|
+-----------+------------------+------------------+------------+------------+



...

-------------------------------------------
Batch: 7
-------------------------------------------
+-----------+------------------+------------------+------------+------------+
|      label|        prediction|          residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+------------------+------------+------------+
|16956.14458| 18711.23116725231| -1755.08658725231| 21257.14286| 33940.25316|
|13253.25228|13171.259451756703| 81.99282824329748| 24363.48548|  35259.5186|
|11724.25532|  9918.92752157049| 1805.327798429511| 14930.29046| 25743.54486|
|10467.46988|13108.626706318992|-2641.156826318991| 25880.57851| 31033.84615|
|15192.28916|15123.150279694102| 69.13888030589806| 25954.95868| 32338.46154|
+-----------+------------------+------------------+------------+------------+



-------------------------------------------
Batch: 8
-------------------------------------------
+-----------+------------------+------------------+------------+------------+
|      label|        prediction|          residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+------------------+------------+------------+
|37479.12226| 33397.68149957353|4081.4407604264634| 33255.33263| 49078.26859|
|35510.97179|33331.483767264675| 2179.488022735328| 32582.47096| 49225.30522|
|8467.841945|   5904.0356569756|2563.8062880243997| 11513.27801| 22636.67396|
|38590.79498|32103.103054521674| 6487.691925478324| 27353.16456| 41161.99336|
|24045.14107|30586.888033776777|-6541.746963776775| 26477.29673| 45581.35405|
+-----------+------------------+------------------+------------+------------+



-------------------------------------------
Batch: 9
-------------------------------------------
+-----------+------------------+-------------------+------------+------------+
|      label|        prediction|           residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+-------------------+------------+------------+
| 10598.8664|12502.672979701203|-1903.8065797012023| 15685.44892| 24336.78689|
|15336.86747|12777.926710658743|  2558.940759341256| 23823.96694| 28843.07692|
|30306.27615| 30900.40317869635| -594.1270286963481| 25477.21519| 36607.57475|
|10829.03226|11581.299413256273| -752.2671532562726| 14721.95122| 22868.42553|
|28381.09091|25767.399835049888|  2613.691074950111| 21610.99796| 42198.66523|
+-----------+------------------+-------------------+------------+------------+



-------------------------------------------
Batch: 10
-------------------------------------------
+-----------+------------------+-------------------+------------+------------+
|      label|        prediction|           residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+-------------------+------------+------------+
|15334.91457| 13918.95624570025| 1415.9583242997505| 21140.42553| 28311.86441|
|17453.49398| 16418.14766553047| 1035.3463144695306| 28223.55372| 34073.84615|
|25516.31799|27433.975923381902|-1917.6579333819027| 26001.26582|  36633.0897|
| 14434.6988|15766.787193186057|-1332.0883931860571| 27334.71074| 33156.92308|
|11403.87097|11366.138622208793|  37.73234779120685| 12106.09756| 22954.21277|
+-----------+------------------+-------------------+------------+------------+



-------------------------------------------
Batch: 11
-------------------------------------------
+-----------+------------------+-------------------+------------+------------+
|      label|        prediction|           residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+-------------------+------------+------------+
|18411.01215|18119.471266031094|  291.5408839689044| 21826.62539| 34685.90164|
|12704.68085|12932.249406505698|-227.56855650569742| 21939.83402| 33828.97155|
|14887.74194| 15175.72582923446|-287.98388923445964| 18943.90244| 30760.85106|
|25262.54545| 22081.03403285577|  3181.511417144233| 19939.30754| 36407.66416|
|17170.12048|15936.085992283064| 1234.0344877169373|  16511.8541| 25828.86076|
+-----------+------------------+-------------------+------------+------------+



In [45]:
query.stop()

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+------------+------------+
|      label|        prediction|          residual|Power_Zone_2|Power_Zone_1|
+-----------+------------------+------------------+------------+------------+
|14232.28916|13680.904451505672| 551.3847084943282| 13798.17629|  22584.3038|
|32493.38912| 28595.96413174785| 3897.424988252151| 26551.89873|  38961.3289|
|13603.40426|12727.824695696234| 875.5795643037654| 25013.27801| 35209.10284|
|10825.53191|  9991.24276056298| 834.2891494370197| 16528.63071| 25579.69365|
|18170.18182|18803.426222876664|-633.2444028766622| 20936.45621| 33642.36814|
+-----------+------------------+------------------+------------+------------+



We stop the query and are finished with observing our streaming data.